# Aula 09 — Schemas múltiplos: API enxuta, cozinha completa

**Objetivo:** desacoplar o que ENTRA/SAI do grafo do que CIRCULA dentro dele.
Ao final você deve saber usar estado privado e `input_schema`/`output_schema`
— e nunca mais vazar `comanda` internas na conta.

**Pré-requisitos:** Aula 02 (canais) e Aula 07 (schemas e acesso).

**Como usar:** rode em ordem. Autocontido, 100% offline (sem LLM, sem chave).
Comece pela CÉLULA 0. **Última aula do bloco 00–09.**

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph
print('dependências prontas.')

## 1. Analogia: salão, cozinha e conta do restaurante

O cliente vê o **cardápio** (entrada) e recebe a **conta** (saída). A **receita
secreta** circula só na cozinha: entra num prato, sai noutro, nunca chega à mesa.

| Restaurante | No LangGraph | O que é de verdade |
|---|---|---|
| cardápio (o que o cliente entrega) | **`input_schema`** | chaves aceitas no `invoke` |
| conta (o que o cliente recebe) | **`output_schema`** | chaves devolvidas no `invoke` |
| receita secreta (só na cozinha) | **estado privado** | schema usado entre nós, filtrado da saída |
| cozinha completa | **schema geral** | todas as chaves que circulam por dentro |

Dois mecanismos independentes, mesmo objetivo: **esconder o interior**.

In [ ]:
# CÉLULA 1 — estado privado: `acrescimo` viaja, não sai (rode e confira)
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class ContaGeral(TypedDict):
    subtotal: int

class TaxaPrivada(TypedDict):
    acrescimo: int

def calcula_taxa(estado: ContaGeral) -> TaxaPrivada:   # ESCREVE no privado
    return {'acrescimo': estado['subtotal'] * 10}

def fecha_conta(estado: TaxaPrivada) -> ContaGeral:   # LÊ o privado
    return {'subtotal': estado['acrescimo'] + 1}

b = StateGraph(ContaGeral)
b.add_node('calcula_taxa', calcula_taxa); b.add_node('fecha_conta', fecha_conta)
b.add_edge(START, 'calcula_taxa'); b.add_edge('calcula_taxa', 'fecha_conta'); b.add_edge('fecha_conta', END)
privado = b.compile()

print('stream (acrescimo existe NO MEIO):')
for passo in privado.stream({'subtotal': 1}):
    print('  ', passo)
out = privado.invoke({'subtotal': 1})
print('invoke (saída final):', out)
assert out == {'subtotal': 11} and 'acrescimo' not in out
print('OK — acrescimo viajou entre os nós e sumiu da saída.')

## 2. Conceito: quem escreve onde (a regra de ouro)

- **Anotação de parâmetro** (`def fecha_conta(estado: TaxaPrivada)`): diz quais
  canais o nó **lê**. O nó enxerga SÓ essas chaves — ler fora dá `KeyError`.
- **Anotação de retorno** (`-> TaxaPrivada`): diz em quais canais o nó
  **escreve**. Valor escrito em canal fora do schema geral viaja, mas é
  filtrado da saída final.
- **`input_schema`**: filtra o que ENTRA (`invoke` aceita só essas chaves;
  extra é ignorado).
- **`output_schema`**: filtra o que SAI (só essas chaves no retorno).
- **Nomes novos**: use `input_schema=`/`output_schema=`. O curso usa
  `input=`/`output=`, que funcionam mas emitem `LangGraphDeprecatedSinceV05`
  (remoção prevista na v2.0).

In [ ]:
# CÉLULA 2 — input/output schemas: entra pedido, sai conta
class PedidoEntrada(TypedDict):
    pedido: str

class ContaSaida(TypedDict):
    conta: str

class Completo(TypedDict):
    pedido: str
    conta: str
    comanda: str          # segredo interno: nunca sai

def pensa(estado: Completo) -> dict:
    return {'conta': 'R$ 14', 'comanda': 'mesa=12'}

def responde(estado: Completo) -> dict:
    return {'conta': estado['conta'] + ' (fechada)'}

b2 = StateGraph(Completo, input_schema=PedidoEntrada, output_schema=ContaSaida)
b2.add_node('pensa', pensa); b2.add_node('responde', responde)
b2.add_edge(START, 'pensa'); b2.add_edge('pensa', 'responde'); b2.add_edge('responde', END)
filtrado = b2.compile()

out = filtrado.invoke({'pedido': '2 pastéis'})
print('com filtro:', out)
assert out == {'conta': 'R$ 14 (fechada)'} and 'comanda' not in out and 'pedido' not in out
print('OK — comanda circularam por dentro e sumiram da saída.')

## 3. Dissecando as duas células

- **CÉLULA 1**: `calcula_taxa` lê `subtotal` (geral) e devolve `acrescimo` (privado). `fecha_conta` lê
  `acrescimo` (privado) e devolve `subtotal` (geral). O `stream` prova que `acrescimo=10`
  existiu no meio; o `invoke` prova que não saiu (`{'subtotal': 11}`).
- **CÉLULA 2**: o grafo interno usa `Completo` (3 chaves), mas o `invoke`
  aceita `{'pedido': ...}` e devolve `{'conta': ...}`. `comanda` viveu
  e morreu dentro da execução.
- **Filtro ≠ validação**: `input_schema` não barra `invoke({})` vazio nem
  exige tipos — ele só *seleciona* chaves. Validação é Pydantic (Aula 07).

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Sem filtro, tudo vaza (o contraste)

O mesmo grafo sem `input_schema`/`output_schema` devolve as 3 chaves.
É assim que `comanda` internas vazam em produção quando ninguém filtra.

In [ ]:
# CÉLULA 3 — sem filtro: tudo vaza (100% offline)
b3 = StateGraph(Completo)
b3.add_node('pensa', pensa); b3.add_node('responde', responde)
b3.add_edge(START, 'pensa'); b3.add_edge('pensa', 'responde'); b3.add_edge('responde', END)
out = b3.compile().invoke({'pedido': '2 pastéis'})
print('sem filtro:', out)
assert out == {'pedido': '2 pastéis', 'conta': 'R$ 14 (fechada)', 'comanda': 'mesa=12'}
print('OK — sem filtro, até segredo interno sai na conta.')

### 4b. Nó lendo fora do próprio schema: `KeyError`

A armadilha que eu mesmo caí pesquisando esta aula: `fecha_conta(estado: Privado)`
tentando ler `estado['subtotal']` (que é do Geral). O nó enxerga SÓ o schema
declarado no parâmetro — o resto não existe para ele.

In [ ]:
# CÉLULA 4 — leitura fora do schema (100% offline)
def fecha_quebrada(estado: TaxaPrivada) -> ContaGeral:
    return {'subtotal': estado['subtotal'] + 1}   # 'subtotal' NÃO está na TaxaPrivada!

bq = StateGraph(ContaGeral)
bq.add_node('calcula_taxa', calcula_taxa); bq.add_node('fecha_conta', fecha_quebrada)
bq.add_edge(START, 'calcula_taxa'); bq.add_edge('calcula_taxa', 'fecha_conta'); bq.add_edge('fecha_conta', END)
try:
    bq.compile().invoke({'subtotal': 1})
    print('rodou?! (não deveria)')
except KeyError as erro:
    print('KeyError:', erro)
    print('OK — cada nó enxerga só o schema declarado no parâmetro.')

### 4c. Chave extra no `invoke` é ignorada

O filtro de entrada descarta o que não está no `input_schema` — inclusive
tentativa de injetar `comanda` por fora.

In [ ]:
# CÉLULA 5 — injeção ignorada (100% offline)
out = filtrado.invoke({'pedido': '2 pastéis', 'comanda': 'inject?'})
print(out)
assert out == {'conta': 'R$ 14 (fechada)'}
print('OK — o filtro barrou a injeção; as comanda reais vieram do nó.')

## 5. Erros clássicos

1. **Ler fora do schema do nó** (CÉLULA 4). Anotação de parâmetro = o que o nó
   enxerga. Precisa de outra chave? Declare no schema do parâmetro.
2. **Achar que filtro valida.** `input_schema` não exige chave nem checa tipo
   (`invoke({})` vazio pode passar se nenhum nó ler a falta). Validação é Aula 07.
3. **Usar `input=`/`output=` sem saber.** Funcionam, mas com aviso de
   depreciação (remoção na v2.0). Prefira `input_schema=`/`output_schema=`.
4. **Esconder o que o próximo nó precisa.** Privado demais quebra a cadeia:
   se `outro_no` precisar de `acrescimo`, ele deve declarar o schema privado também.
5. **Vazar segredo por falta de filtro** (CÉLULA 3). Regra de produção: todo
   grafo com dado interno tem `output_schema` explícito.
6. **Confundir privado com memória.** Estado privado some no fim do `invoke`;
   atravessar turnos é `checkpointer` (Aula 06), não schema.

## 6. Exercício guiado — avaliador com rubrica secreta

Situação (ponte direta para o seu avaliador de redação): o grafo recebe
`texto`, consulta a `rubrica` interna (privada) e devolve só `nota`.
A rubrica nunca aparece na saída — como gabarito que o aluno não vê.

In [ ]:
# CÉLULA 6 — exercício guiado (rode e confira o segredo escondido)
class Avaliacao(TypedDict):
    texto: str
    nota: float
    rubrica: str       # segredo interno

class PedidoNota(TypedDict):
    texto: str

class Boletim(TypedDict):
    nota: float

def avalia(estado: Avaliacao) -> dict:
    # finge a rubrica: texto longo + palavra-chave = nota alta
    rubrica = 'clareza: 40%, exemplos: 30%, conclusao: 30%'
    pontos = 4.0 + (2.0 if 'exemplo' in estado['texto'] else 0.0)
    pontos += 2.0 if len(estado['texto']) > 50 else 0.0
    return {'nota': min(pontos, 10.0), 'rubrica': rubrica}

ba = StateGraph(Avaliacao, input_schema=PedidoNota, output_schema=Boletim)
ba.add_node('avalia', avalia)
ba.add_edge(START, 'avalia'); ba.add_edge('avalia', END)
avaliador = ba.compile()

r1 = avaliador.invoke({'texto': 'Redação longa com exemplo concreto de aplicação. ' * 3})
r2 = avaliador.invoke({'texto': 'curta'})
print(r1, '|', r2)
assert r1 == {'nota': 8.0} and r2 == {'nota': 4.0}, (r1, r2)
assert 'rubrica' not in r1 and 'texto' not in r1
print('OK — nota sai, rubrica e texto não vazam.')

## 7. Desafio — 2 etapas com segredo entre elas

Amplie o avaliador: `rascunho` (nó 1 gera nota parcial + `justificativa`
privada) → `revisao` (nó 2 lê a justificativa e ajusta a nota) → saída só
`{'nota': ...}`.

- PedidoEntrada: `{'texto': ...}`. Saída: `{'nota': ...}` (nada mais).
- A `justificativa` viaja do nó 1 ao nó 2 via estado privado e some no final.
- Valide com 2 textos (um que passa, um que não) e confira que nem `texto`,
  nem `rubrica`, nem `justificativa` aparecem na saída.
- **Pegadinha:** o nó 2 precisa declarar o schema privado no parâmetro —
  senão, `KeyError` (CÉLULA 4).

## 8. Checklist + FECHAMENTO DO BLOCO 00–09

**Você entendeu se consegue responder sem olhar:**
- [ ] O que a anotação de parâmetro do nó controla? E a de retorno?
- [ ] Para que servem `input_schema` e `output_schema` — e por que evitar `input=`/`output=`?
- [ ] Onde o `acrescimo` existe: no stream, no invoke, nos dois, em nenhum?
- [ ] Filtro valida tipo e exige chave? (Se não, quem valida?)
- [ ] No seu avaliador de redação, o que seria entrada, privado e saída?

**O bloco 00–09 em uma frase por aula:**
- 00: grafo = fluxograma com estado compartilhado (lanchonete).
- 01: ambiente + 1ª chamada LLM (`invoke`/`stream`, `temperature`, raciocínio).
- 02: nós, arestas, condicional, `compile`/`invoke`, overwrite.
- 03: mensagens como estado, `bind_tools` pede sem executar, `add_messages`.
- 04: `tools_condition` + `ToolNode` executam; `ToolMessage` é o recibo.
- 05: a volta `tools → assistant` cria o agente ReAct.
- 06: `checkpointer` + `thread_id` viram invokes isolados em conversa.
- 07: `TypedDict`/hint vs `dataclass` vs Pydantic/validação.
- 08: paralelo exige reducer; `add_messages` tem 3 poderes.
- 09: privado viaja e some; `input_schema`/`output_schema` blindam a API.

**Próximo bloco (módulos 3–4):** filtrar/cortar mensagens, streaming +
interrupção + humano-no-loop, breakpoints, viagem no tempo, paralelização,
sub-grafos e MapReduce — tudo construído sobre estas 10 aulas.